Cell 0 — setup (run at the start of every session)

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

BASE = "/content/drive/MyDrive/norris/outputs"
OUT  = f"{BASE}/generations/baselines"
for d in [OUT, f"{BASE}/logs", f"{BASE}/tables"]:
    os.makedirs(d, exist_ok=True)

FLORES_REV  = "5fec6c13f9e5a4db2f745d4ec0d7c9721ddc4f06"
FLORES_CODE = {"eng_Latn": "eng_Latn", "spa_Latn": "spa_Latn", "tgl_Latn": "fil_Latn"}  # NLLB code -> FLORES+ config
LANGS = list(FLORES_CODE)
DIRECTIONS = [("eng_Latn","tgl_Latn"), ("tgl_Latn","eng_Latn"),
              ("eng_Latn","spa_Latn"), ("spa_Latn","eng_Latn")]
print("setup OK")

Mounted at /content/drive
setup OK


Cell 1 — load FLORES+ and save manifest (DONE, don't run now)

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=3.0" sentencepiece
from huggingface_hub import login; login()
import hashlib, json, sys, datetime, torch, transformers, datasets
from datasets import load_dataset

data, manifest = {}, {}
for split in ["dev", "devtest"]:
    for lg in LANGS:
        d = load_dataset("openlanguagedata/flores_plus", FLORES_CODE[lg], split=split, revision=FLORES_REV)
        rows = sorted(zip(d["id"], d["text"]))
        data[(split, lg)] = rows
        blob = "\n".join(t for _, t in rows).encode("utf-8")
        manifest[f"{split}/{lg}"] = {"flores_plus_config": FLORES_CODE[lg], "n": len(rows),
                                     "sha256": hashlib.sha256(blob).hexdigest()}
        print(split, lg, "<-", FLORES_CODE[lg], len(rows))

for split in ["dev", "devtest"]:
    ids = [[i for i, _ in data[(split, lg)]] for lg in LANGS]
    assert ids[0] == ids[1] == ids[2], f"ID misalignment in {split}"
print("alignment OK")

json.dump({
    "dataset": "openlanguagedata/flores_plus", "revision": FLORES_REV,
    "license": "CC-BY-SA-4.0 (gated; terms accepted 2026-09-20)",
    "checksum_method": "sha256 over UTF-8 text, rows sorted by FLORES id, joined with \\n",
    "code_mapping_note": "NLLB tgl_Latn <- FLORES+ fil_Latn",
    "created_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "env": {"python": sys.version.split()[0], "torch": torch.__version__,
            "transformers": transformers.__version__, "datasets": datasets.__version__,
            "gpu": torch.cuda.get_device_name(0)},
    "splits": manifest,
}, open(f"{BASE}/logs/data_manifest.json", "w"), indent=2)

for i in [0, 500]:   # manual inspection sample
    for lg in LANGS:
        print(lg, "|", data[("dev", lg)][i][1][:150])
    print("---")

Cell 2 — NLLB generation

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

CKPT = "facebook/nllb-200-3.3B"
model = AutoModelForSeq2SeqLM.from_pretrained(CKPT, dtype=torch.float16).cuda().eval()

# smoke test: one sentence per direction
for src, tgt in DIRECTIONS:
    tok = AutoTokenizer.from_pretrained(CKPT, src_lang=src)
    enc = tok([data[("dev", src)][1][1]], return_tensors="pt").to("cuda")
    out = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(tgt), num_beams=4, max_length=512)
    print(f"{src}->{tgt}:", tok.batch_decode(out, skip_special_tokens=True)[0][:150])

def run(src, tgt, split, bs=32):
    path = f"{OUT}/nllb3.3b_{src}_{tgt}_{split}.jsonl"
    done = set()
    if os.path.exists(path):
        done = {json.loads(l)["sentence_id"] for l in open(path, encoding="utf-8")}
    tok = AutoTokenizer.from_pretrained(CKPT, src_lang=src)
    bos = tok.convert_tokens_to_ids(tgt)
    S, R = data[(split, src)], data[(split, tgt)]
    todo = [(sid, s, r) for (sid, s), (_, r) in zip(S, R) if sid not in done]
    t_start = time.time()
    with open(path, "a", encoding="utf-8") as f:
        for i in tqdm(range(0, len(todo), bs), desc=f"{src}->{tgt} {split}"):
            chunk = todo[i:i+bs]
            enc = tok([c[1] for c in chunk], return_tensors="pt",
                      padding=True, truncation=True, max_length=512).to("cuda")
            t0 = time.time()
            with torch.inference_mode():
                out = model.generate(**enc, forced_bos_token_id=bos, num_beams=4,
                                     max_length=512, length_penalty=1.0, early_stopping=True)
            dt = (time.time() - t0) * 1000 / len(chunk)
            for (sid, s, r), h in zip(chunk, tok.batch_decode(out, skip_special_tokens=True)):
                f.write(json.dumps({"sentence_id": sid, "source_text": s, "reference_text": r,
                    "nllb_translation": h, "generation_time_ms": round(dt, 2),
                    "token_count": len(tok(h, add_special_tokens=False).input_ids)},
                    ensure_ascii=False) + "\n")
            f.flush()
    print(f"{src}->{tgt} {split}: {len(todo)} new rows, {time.time()-t_start:.0f}s")

for split in ["devtest", "dev"]:
    for src, tgt in DIRECTIONS:
        run(src, tgt, split)

Cell 3 — quality control

In [ ]:
!pip -q install langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
LID = {"eng_Latn": "en", "spa_Latn": "es", "tgl_Latn": "tl"}

def lid(t):
    try: return detect(t)
    except Exception: return "unk"

qc, outliers = {}, {}
for split in ["devtest", "dev"]:
    for src, tgt in DIRECTIONS:
        rows = [json.loads(l) for l in open(f"{OUT}/nllb3.3b_{src}_{tgt}_{split}.jsonl", encoding="utf-8")]
        ids = [r["sentence_id"] for r in rows]
        expected = [i for i, _ in data[(split, src)]]
        ratios = [len(r["nllb_translation"]) / max(len(r["reference_text"]), 1) for r in rows]
        key = f"{split}/{src}->{tgt}"
        qc[key] = {
            "rows": len(rows), "expected": len(expected),
            "ids_match": sorted(ids) == sorted(expected),
            "duplicates": len(ids) - len(set(ids)),
            "empty": sum(not r["nllb_translation"].strip() for r in rows),
            "copied_source": sum(r["nllb_translation"].strip() == r["source_text"].strip() for r in rows),
            "len_ratio_outliers": sum(x < 0.5 or x > 2.0 for x in ratios),
            "hyp_in_target_lang": round(sum(lid(r["nllb_translation"]) == LID[tgt] for r in rows) / len(rows), 3),
            "ref_in_target_lang": round(sum(lid(r["reference_text"]) == LID[tgt] for r in rows) / len(rows), 3),
        }
        outliers[key] = [(r["sentence_id"], round(x, 2), r["nllb_translation"][:120])
                         for r, x in zip(rows, ratios) if x < 0.5 or x > 2.0][:3]
        print(key, qc[key])

json.dump({"qc": qc, "outlier_samples": outliers}, open(f"{BASE}/logs/qc_baselines.json", "w"),
          indent=2, ensure_ascii=False)

Cell 4 — check everything is on Drive

In [2]:
import glob
for p in sorted(glob.glob(f"{OUT}/*.jsonl")):
    n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{os.path.basename(p):45s} {n:5d} lines")
print()
for p in sorted(glob.glob(f"{BASE}/logs/*")):
    print("log:", os.path.basename(p))

nllb3.3b_eng_Latn_spa_Latn_dev.jsonl            997 lines
nllb3.3b_eng_Latn_spa_Latn_devtest.jsonl       1012 lines
nllb3.3b_eng_Latn_tgl_Latn_dev.jsonl            997 lines
nllb3.3b_eng_Latn_tgl_Latn_devtest.jsonl       1012 lines
nllb3.3b_spa_Latn_eng_Latn_dev.jsonl            997 lines
nllb3.3b_spa_Latn_eng_Latn_devtest.jsonl       1012 lines
nllb3.3b_tgl_Latn_eng_Latn_dev.jsonl            997 lines
nllb3.3b_tgl_Latn_eng_Latn_devtest.jsonl       1012 lines

log: data_manifest.json
log: qc_baselines.json


Cell 5 — build the separate evaluation environment

In [8]:
!pip -q install uv
!uv venv --clear --python 3.11 /content/evalenv
!uv pip install --python /content/evalenv/bin/python "torch==2.6.0" --index-url https://download.pytorch.org/whl/cu124
!uv pip install --python /content/evalenv/bin/python "unbabel-comet>=2.2,<3" "transformers<4.50" "huggingface-hub<1" "numpy<2" "sacrebleu>=2.4" "setuptools<81"
!/content/evalenv/bin/python -c "import torch, comet; print('cuda:', torch.cuda.is_available())"

Using CPython 3.11.16
Creating virtual environment at: evalenv
Activate with: source evalenv/bin/activate
Using Python 3.11.16 environment at: evalenv
Resolved 23 packages in 145ms
Installed 23 packages in 261ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4.127
 + nvidia-cuda-nvrtc-cu12==12.4.127
 + nvidia-cuda-runtime-cu12==12.4.127
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.2.1.3
 + nvidia-curand-cu12==10.3.5.147
 + nvidia-cusolver-cu12==11.6.1.9
 + nvidia-cusparse-cu12==12.3.1.170
 + nvidia-cusparselt-cu12==0.6.2
 + nvidia-nccl-cu12==2.21.5
 + nvidia-nvjitlink-cu12==12.4.127
 + nvidia-nvtx-cu12==12.4.127
 + sympy==1.13.1
 + torch==2.6.0+cu124
 + triton==3.2.0
 + typing-extensions==4.16.0
Using Python 3.11.16 environment at: evalenv
Resolved 64 packages in 22ms
Installed 41 packages in 52ms
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + ai

In [9]:
!uv pip install --python /content/evalenv/bin/python "setuptools<81"
!/content/evalenv/bin/python -c "import torch, comet; print('cuda:', torch.cuda.is_available())"

Using Python 3.11.16 environment at: evalenv
Checked 1 package in 2ms
/content/evalenv/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
cuda: True


Cell 6 — write the scoring script

In [4]:
%%writefile /content/score_baseline.py
import json, sys, os, numpy as np
from importlib.metadata import version
from sacrebleu.metrics import BLEU, CHRF
from comet import download_model, load_from_checkpoint

BASE = "/content/drive/MyDrive/norris/outputs"
OUT = f"{BASE}/generations/baselines"
SPLIT = "dev"  # devtest stays unscored until the Phase 3 protocol freeze
DIRECTIONS = [("eng_Latn","tgl_Latn"), ("tgl_Latn","eng_Latn"),
              ("eng_Latn","spa_Latn"), ("spa_Latn","eng_Latn")]

env = {p: version(p) for p in ["torch", "transformers", "unbabel-comet", "sacrebleu", "numpy"]}
env["python"] = sys.version.split()[0]
print("EVAL ENV:", env, flush=True)

cm = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
bleu, chrf = BLEU(tokenize="13a"), CHRF(word_order=2)
rng = np.random.default_rng(0)
res = {}

for src, tgt in DIRECTIONS:
    rows = sorted((json.loads(l) for l in open(f"{OUT}/nllb3.3b_{src}_{tgt}_{SPLIT}.jsonl", encoding="utf-8")),
                  key=lambda r: r["sentence_id"])
    hyp = [r["nllb_translation"] for r in rows]
    ref = [[r["reference_text"] for r in rows]]
    b = bleu.corpus_score(hyp, ref, n_bootstrap=1000)
    c = chrf.corpus_score(hyp, ref, n_bootstrap=1000)
    out = cm.predict([{"src": r["source_text"], "mt": r["nllb_translation"], "ref": r["reference_text"]}
                      for r in rows], batch_size=64, gpus=1, progress_bar=False)
    seg = np.array(out.scores)
    boot = [seg[rng.integers(0, len(seg), len(seg))].mean() for _ in range(1000)]
    json.dump(dict(zip([r["sentence_id"] for r in rows], out.scores)),
              open(f"{BASE}/logs/comet_seg_nllb_{src}_{tgt}_{SPLIT}.json", "w"))
    res[f"{src}->{tgt}"] = {
        "n": len(rows),
        "bleu": round(b.score, 2), "bleu_ci": round(getattr(b, "_ci", 0), 2),
        "chrf++": round(c.score, 2), "chrf_ci": round(getattr(c, "_ci", 0), 2),
        "comet": round(seg.mean() * 100, 2),
        "comet_ci": round((np.percentile(boot, 97.5) - np.percentile(boot, 2.5)) / 2 * 100, 2),
    }
    print("RESULT", src, "->", tgt, res[f"{src}->{tgt}"], flush=True)

print("BLEU SIG:", bleu.get_signature())
print("CHRF SIG:", chrf.get_signature())
json.dump({"env": env, "results": res}, open(f"{BASE}/logs/nllb_baseline_metrics_{SPLIT}.json", "w"), indent=2)

N = {"eng_Latn": "En", "tgl_Latn": "Tl", "spa_Latn": "Es"}
tex = [r"\begin{table}[t]", r"\centering", r"\small", r"\begin{tabular}{lrrr}", r"\toprule",
       r"Direction & BLEU & chrF++ & COMET \\", r"\midrule"]
for k, v in res.items():
    s, t = k.split("->")
    tex.append(f"{N[s]}$\\rightarrow${N[t]} & {v['bleu']:.1f}$\\pm${v['bleu_ci']:.1f} & "
               f"{v['chrf++']:.1f}$\\pm${v['chrf_ci']:.1f} & {v['comet']:.1f}$\\pm${v['comet_ci']:.1f} \\\\")
tex += [r"\bottomrule", r"\end{tabular}",
        r"\caption{NLLB-200 3.3B baseline on FLORES+ dev ($n{=}997$ per direction). "
        r"$\pm$: 95\% bootstrap CI half-width, 1{,}000 resamples. Devtest held out.}",
        r"\label{tab:nllb-baseline-dev}", r"\end{table}"]
os.makedirs(f"{BASE}/tables", exist_ok=True)
open(f"{BASE}/tables/nllb_baseline_dev.tex", "w").write("\n".join(tex))
print("\n".join(tex))

Writing /content/score_baseline.py


Cell 7 — run scoring

In [10]:
!/content/evalenv/bin/python /content/score_baseline.py

/content/evalenv/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
EVAL ENV: {'torch': '2.6.0+cu124', 'transformers': '4.49.0', 'unbabel-comet': '2.2.7', 'sacrebleu': '2.6.0', 'numpy': '1.26.4', 'python': '3.11.16'}






Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`




Encoder model frozen.
/content/evalenv/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:216: Found key

In [11]:
from huggingface_hub import HfApi
import json, random
print("NLLB revision:", HfApi().model_info("facebook/nllb-200-3.3B").sha)
random.seed(0)
for src, tgt in DIRECTIONS:
    rows = [json.loads(l) for l in open(f"{OUT}/nllb3.3b_{src}_{tgt}_dev.jsonl", encoding="utf-8")]
    for r in random.sample(rows, 5):
        print(f"[{src}->{tgt} #{r['sentence_id']}]\nSRC:  {r['source_text']}\nNLLB: {r['nllb_translation']}\nREF:  {r['reference_text']}\n")

NLLB revision: 1a07f7d195896b2114afcb79b7b57ab512e7b43e
[eng_Latn->tgl_Latn #864]
SRC:  Thus the scourges of Delhi Belly, the Pharaoh's Curse, Montezuma's Revenge, and their many friends.
NLLB: Kaya ang mga salot ng Delhi Belly, ang Sumpa ng Paraon, ang Paghihiganti ni Montezuma, at ang kanilang maraming mga kaibigan.
REF:  Kaya mayroong mga parusang Tiyan ng Delhi, ang Sumpa ng Paraon, Paghihiganti ni Montezuma, at ang marami nilang kaibigan.

[eng_Latn->tgl_Latn #394]
SRC:  But the Royal Navy was still much stronger than the German Navy (“Kriegsmarine”) and could have destroyed any invasion fleet sent across the English Channel.
NLLB: Ngunit mas malakas pa rin ang Royal Navy kaysa sa German Navy (Kriegsmarine) at maaaring sirain ang anumang armada ng pagsakop na ipinadala sa English Channel.
REF:  Ngunit higit na malakas pa rin ang Royal Navy kumpara sa German Navy ("Kriegsmarine") at makakayang wasakin ang anumang armadang pansalakay sa buong English Channel.

[eng_Latn->tgl_Latn #7

In [14]:
# PHASE 2 DIAGNOSTICS (dev only) — complete, self-contained
!pip -q install sacrebleu
import json, re, numpy as np
from collections import Counter
from sacrebleu.metrics import BLEU

NLLB_REV = "1a07f7d195896b2114afcb79b7b57ab512e7b43e"
bleu = BLEU(tokenize="13a")
WORD = re.compile(r"[^\W\d_]{4,}")
NUM = re.compile(r"\d+")

def load(src, tgt):
    rows = sorted((json.loads(l) for l in open(f"{OUT}/nllb3.3b_{src}_{tgt}_dev.jsonl", encoding="utf-8")),
                  key=lambda r: r["sentence_id"])
    seg = json.load(open(f"{BASE}/logs/comet_seg_nllb_{src}_{tgt}_dev.json"))
    for r in rows:
        r["comet"] = seg[str(r["sentence_id"])]
    return rows

def copy_rate(eng, other):
    e = {w.lower() for w in WORD.findall(eng)}
    o = [w.lower() for w in WORD.findall(other)]
    return sum(w in e for w in o) / len(o) if o else 0.0

def missing(src, out):
    return bool(Counter(NUM.findall(src)) - Counter(NUM.findall(out)))

R = {f"{s}->{t}": load(s, t) for s, t in DIRECTIONS}
diag = {}

# A. HEADROOM
print("A. HEADROOM (segment COMET, dev)")
for k, rows in R.items():
    s = np.array([r["comet"] for r in rows])
    h = np.histogram(s, bins=[0, .70, .80, .90, 1.01])[0] / len(s) * 100
    diag.setdefault("headroom", {})[k] = {"median": round(float(np.median(s)), 4), "sd": round(float(s.std()), 4),
        "<0.70": round(float(h[0]), 1), "0.70-0.80": round(float(h[1]), 1),
        "0.80-0.90": round(float(h[2]), 1), ">=0.90": round(float(h[3]), 1)}
    print(k, diag["headroom"][k])

# B1. ENGLISH-WORD COPY RATE
print("\nB1. ENGLISH-WORD COPY RATE")
for k in ["eng_Latn->tgl_Latn", "eng_Latn->spa_Latn"]:
    rows = R[k]
    diag.setdefault("copy_rate", {})[k] = {
        "reference": round(float(np.mean([copy_rate(r["source_text"], r["reference_text"]) for r in rows])), 4),
        "nllb": round(float(np.mean([copy_rate(r["source_text"], r["nllb_translation"]) for r in rows])), 4)}
    print(k, diag["copy_rate"][k])

# B2. INTO-ENGLISH SCORES BY COPY-RATE GROUP
print("\nB2. INTO-ENGLISH SCORES BY COPY-RATE GROUP")
for k in ["tgl_Latn->eng_Latn", "spa_Latn->eng_Latn"]:
    rows = R[k]
    cr = np.array([copy_rate(r["reference_text"], r["source_text"]) for r in rows])
    med = np.median(cr[cr > 0])
    for name, mask in {"none": cr == 0, "some": (cr > 0) & (cr <= med), "high": cr > med}.items():
        sub = [r for r, m in zip(rows, mask) if m]
        if not sub:
            print(k, name, "empty"); continue
        b = bleu.corpus_score([r["nllb_translation"] for r in sub], [[r["reference_text"] for r in sub]]).score
        diag.setdefault("copy_groups", {}).setdefault(k, {})[name] = {"n": len(sub),
            "mean_copy": round(float(cr[mask].mean()), 3), "bleu": round(b, 1),
            "comet": round(float(np.mean([r["comet"] for r in sub]) * 100), 1)}
        print(k, name, diag["copy_groups"][k][name])

# C. NUMBER PRESERVATION
print("\nC. NUMBER MISMATCH")
for k, rows in R.items():
    wn = [r for r in rows if NUM.search(r["source_text"])]
    diag.setdefault("numbers", {})[k] = {"sentences_with_numbers": len(wn),
        "nllb_mismatch_pct": round(float(100 * np.mean([missing(r["source_text"], r["nllb_translation"]) for r in wn])), 1),
        "reference_mismatch_pct": round(float(100 * np.mean([missing(r["source_text"], r["reference_text"]) for r in wn])), 1)}
    print(k, diag["numbers"][k])

# D. MINIMUM DETECTABLE EFFECT (planning estimate, devtest n=1012)
print("\nD. MINIMUM DETECTABLE COMET DIFFERENCE (80% power)")
for k, rows in R.items():
    sd = float(np.std([r["comet"] for r in rows]))
    out = {}
    for rho in (0.5, 0.8):
        sdd = sd * np.sqrt(2 * (1 - rho))
        out[f"rho={rho}"] = {"alpha_0.05": round(float((1.96 + .84) * sdd / np.sqrt(1012) * 100), 2),
                             "holm_24_tests": round(float((3.08 + .84) * sdd / np.sqrt(1012) * 100), 2)}
    diag.setdefault("mde", {})[k] = out
    print(k, out)

# E. SAVE + 40 LOWEST-COMET SENTENCES FOR ANNOTATION
low = {k: sorted(rows, key=lambda r: r["comet"])[:10] for k, rows in R.items()}
json.dump({"nllb_revision": NLLB_REV, "diagnostics": diag,
           "lowest10": {k: [{kk: r[kk] for kk in ["sentence_id", "comet", "source_text", "nllb_translation", "reference_text"]}
                            for r in v] for k, v in low.items()}},
          open(f"{BASE}/logs/phase2_diagnostics_dev.json", "w"), indent=2, ensure_ascii=False)
print("\nSaved phase2_diagnostics_dev.json")
for k, v in low.items():
    print(f"\n===== {k} : 10 lowest COMET =====")
    for r in v:
        print(f"#{r['sentence_id']} COMET={r['comet']:.3f}\n SRC:  {r['source_text']}\n NLLB: {r['nllb_translation']}\n REF:  {r['reference_text']}\n")

A. HEADROOM (segment COMET, dev)
eng_Latn->tgl_Latn {'median': 0.8618, 'sd': 0.0719, '<0.70': 4.4, '0.70-0.80': 15.6, '0.80-0.90': 56.7, '>=0.90': 23.3}
tgl_Latn->eng_Latn {'median': 0.8864, 'sd': 0.056, '<0.70': 1.2, '0.70-0.80': 6.5, '0.80-0.90': 54.1, '>=0.90': 38.2}
eng_Latn->spa_Latn {'median': 0.8778, 'sd': 0.0605, '<0.70': 2.4, '0.70-0.80': 11.3, '0.80-0.90': 58.6, '>=0.90': 27.7}
spa_Latn->eng_Latn {'median': 0.8813, 'sd': 0.0446, '<0.70': 0.7, '0.70-0.80': 5.3, '0.80-0.90': 65.8, '>=0.90': 28.2}

B1. ENGLISH-WORD COPY RATE
eng_Latn->tgl_Latn {'reference': 0.098, 'nllb': 0.1414}
eng_Latn->spa_Latn {'reference': 0.0691, 'nllb': 0.0834}

B2. INTO-ENGLISH SCORES BY COPY-RATE GROUP
tgl_Latn->eng_Latn none {'n': 443, 'mean_copy': 0.0, 'bleu': 47.0, 'comet': 87.2}
tgl_Latn->eng_Latn some {'n': 292, 'mean_copy': 0.089, 'bleu': 49.1, 'comet': 87.8}
tgl_Latn->eng_Latn high {'n': 262, 'mean_copy': 0.274, 'bleu': 58.3, 'comet': 88.6}
spa_Latn->eng_Latn none {'n': 538, 'mean_copy': 0.0, 'b

In [15]:
import json, re, numpy as np
SENT = re.compile(r'(?<=[.!?])["»”\']?\s+(?=[A-ZÁÉÍÓÚÑ¿¡«"“])')
def n_sent(t): return len([s for s in SENT.split(t.strip()) if s.strip()])

ms = {}
for s, t in DIRECTIONS:
    k = f"{s}->{t}"
    rows = [json.loads(l) for l in open(f"{OUT}/nllb3.3b_{s}_{t}_dev.jsonl", encoding="utf-8")]
    seg = json.load(open(f"{BASE}/logs/comet_seg_nllb_{s}_{t}_dev.json"))
    multi = [r for r in rows if n_sent(r["source_text"]) > 1]
    single = [r for r in rows if n_sent(r["source_text"]) == 1]
    dropped = [r for r in multi if n_sent(r["nllb_translation"]) < n_sent(r["source_text"])]
    low = [r for r in rows if seg[str(r["sentence_id"])] < 0.80]
    ms[k] = {
        "multi_sentence_items": len(multi), "pct_of_dev": round(100 * len(multi) / len(rows), 1),
        "nllb_dropped_sentences": len(dropped), "pct_of_multi_dropped": round(100 * len(dropped) / max(len(multi), 1), 1),
        "comet_single": round(100 * float(np.mean([seg[str(r["sentence_id"])] for r in single])), 1),
        "comet_multi": round(100 * float(np.mean([seg[str(r["sentence_id"])] for r in multi])), 1),
        "comet_multi_dropped": round(100 * float(np.mean([seg[str(r["sentence_id"])] for r in dropped])), 1) if dropped else None,
        "pct_of_low_pool_that_dropped": round(100 * sum(r in dropped for r in low) / max(len(low), 1), 1),
    }
    print(k, ms[k])

devtest_multi = {s: sum(n_sent(json.loads(l)["source_text"]) > 1
                        for l in open(f"{OUT}/nllb3.3b_{s}_{t}_devtest.jsonl", encoding="utf-8"))
                 for s, t in DIRECTIONS}
print("\ndevtest multi-sentence SOURCE counts (sources only, no scoring):", devtest_multi)
json.dump({"dev": ms, "devtest_source_multi_counts": devtest_multi},
          open(f"{BASE}/logs/multisentence_omission_dev.json", "w"), indent=2)

eng_Latn->tgl_Latn {'multi_sentence_items': 112, 'pct_of_dev': 11.2, 'nllb_dropped_sentences': 39, 'pct_of_multi_dropped': 34.8, 'comet_single': 85.0, 'comet_multi': 82.4, 'comet_multi_dropped': 75.7, 'pct_of_low_pool_that_dropped': 10.0}
tgl_Latn->eng_Latn {'multi_sentence_items': 112, 'pct_of_dev': 11.2, 'nllb_dropped_sentences': 24, 'pct_of_multi_dropped': 21.4, 'comet_single': 87.9, 'comet_multi': 86.6, 'comet_multi_dropped': 82.1, 'pct_of_low_pool_that_dropped': 6.5}
eng_Latn->spa_Latn {'multi_sentence_items': 112, 'pct_of_dev': 11.2, 'nllb_dropped_sentences': 65, 'pct_of_multi_dropped': 58.0, 'comet_single': 86.5, 'comet_multi': 83.6, 'comet_multi_dropped': 81.6, 'pct_of_low_pool_that_dropped': 16.8}
spa_Latn->eng_Latn {'multi_sentence_items': 128, 'pct_of_dev': 12.8, 'nllb_dropped_sentences': 71, 'pct_of_multi_dropped': 55.5, 'comet_single': 87.6, 'comet_multi': 86.2, 'comet_multi_dropped': 84.9, 'pct_of_low_pool_that_dropped': 18.3}

devtest multi-sentence SOURCE counts (source